# Component 0 QC Notebook

This notebook is only a thin consumer of `src.components.component0_qc`.
The actual preprocessing logic lives in the package code so the notebook stays disposable.

Update the external HDD paths below before running it on your datasets.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

from src.components.component0_qc import harmonise_sample

## External HDD Configuration

Point these paths at the datasets on your external drive when it is connected.

In [ ]:
EXTERNAL_HDD_ROOT = Path(r"E:\\tb_cxr_datasets")

DATASET_ID = "montgomery"
VIEW = "PA"
PIXEL_SPACING_CM = None
USE_SYNTHETIC_IF_MISSING = True

IMAGE_PATH = EXTERNAL_HDD_ROOT / "Montgomery" / "CXR_png" / "MCUCXR_0001_0.png"
IMAGE_ID = IMAGE_PATH.stem if IMAGE_PATH.suffix else "synthetic-demo"

IMAGE_PATH

## Loading Helpers

In [ ]:
def load_grayscale_image(path: Path) -> np.ndarray:
    image = Image.open(path)
    array = np.asarray(image)
    if array.ndim == 3:
        array = array[..., 0]
    return array


def make_synthetic_image(dataset_id: str, size: int = 768) -> np.ndarray:
    max_value = 4095 if dataset_id.lower() == "montgomery" else 255
    x = np.linspace(0, max_value, size, dtype=np.float32)
    image = np.tile(x, (size, 1))
    yy, xx = np.ogrid[:size, :size]
    lesion = (xx - size / 2) ** 2 + (yy - size / 2) ** 2 < (size / 5) ** 2
    image[lesion] *= 0.55
    return image.astype(np.uint16 if max_value > 255 else np.uint8)

## Build A Sample And Run Component 0

In [ ]:
if IMAGE_PATH.exists():
    image = load_grayscale_image(IMAGE_PATH)
    source = "external-file"
else:
    if not USE_SYNTHETIC_IF_MISSING:
        raise FileNotFoundError(f"Missing input image: {IMAGE_PATH}")
    image = make_synthetic_image(DATASET_ID)
    source = "synthetic-fallback"

sample = {
    "image": image,
    "image_id": IMAGE_ID,
    "dataset_id": DATASET_ID,
    "domain_id": 0,
    "view": VIEW,
    "pixel_spacing_cm": PIXEL_SPACING_CM,
    "path": str(IMAGE_PATH),
    "labels": {},
}

harmonised = harmonise_sample(sample)

print(f"source={source}")
print(f"raw-shape={image.shape}, raw-dtype={image.dtype}")

## Inspect The Output Contract

In [ ]:
print("x_1024:", tuple(harmonised.x_1024.shape), harmonised.x_1024.dtype)
print("x_224:", tuple(harmonised.x_224.shape), harmonised.x_224.dtype)
print("x_3ch:", tuple(harmonised.x_3ch.shape), harmonised.x_3ch.dtype)
harmonised.meta

## Visualise The Branches

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(image, cmap="gray")
axes[0].set_title("raw")
axes[0].axis("off")

axes[1].imshow(harmonised.x_1024.squeeze(0).cpu().numpy(), cmap="gray")
axes[1].set_title("x_1024")
axes[1].axis("off")

axes[2].imshow(harmonised.x_224.squeeze(0).cpu().numpy(), cmap="gray")
axes[2].set_title("x_224")
axes[2].axis("off")

axes[3].imshow(harmonised.x_3ch[0].cpu().numpy(), cmap="gray")
axes[3].set_title("x_3ch[0]")
axes[3].axis("off")

plt.tight_layout()

## Optional Artifact Save

In [ ]:
output_dir = REPO_ROOT / "data" / "cache" / "component0_demo"
output_dir.mkdir(parents=True, exist_ok=True)

artifact_path = output_dir / f"{sample['image_id']}_component0.pt"
torch.save(
    {
        "x_1024": harmonised.x_1024,
        "x_224": harmonised.x_224,
        "x_3ch": harmonised.x_3ch,
        "meta": harmonised.meta,
    },
    artifact_path,
)

artifact_path